#**1. PERSIAPAN DATA - LOG TRANSAKSI VENDING MACHINE ANTAR GEDUNG**

Ditahap awal ini, proses cleansing dimulai dari menyiapkan dataset log transaksi vending machine yang sudah di upload ke Google Drive. Saya menggunakan data mentah/kotor sebanyak 50 baris dengan 10 kolom informasi data.

##**1.1 Import library**

Library yang dibutuhkan untuk proses data cleansing ini antara lain :
1. pandas (pd), digunakan untuk membaca file Excel, mengolah data dalam bentuk DataFrame, membersihkan data, melakukan filtering, dan menyimpan kembali dataset bersih.
2. numpy (np), untuk menangani nilai kosong (NaN) selama proses cleansing data.
3. re, digunakan untuk melakukan pencarian dan penghapusan pola atau karakter khusus dalam data teks menggunakan Regular Expression (regex), misalnya saat membersihkan format harga dan tanggal.

In [23]:
import pandas as pd
import numpy as np
import re

##**1.2 Membaca dataset**

Selanjutnya, code dibawah ini digunakan untuk mengambil dataset vending machine dari Google Drive dan menampilkan beberapa baris awal untuk memastikan data berhasil dibaca.

In [24]:
from google.colab import drive

# Menghubungkan Google Colab dengan Google Drive
drive.mount('/content/drive')

# Membaca dataset Excel
file_path = '/content/drive/My Drive/2418014_DATASET DATMIN/dataset_kotor_vending_machine.xlsx'
df = pd.read_excel(file_path)

# Menyimpan kondisi awal dataset untuk perbandingan hasil cleansing
baris_awal = len(df)
missing_awal = df.isnull().sum().sum()
duplikat_awal = df.duplicated().sum()

print("Dataset Log Transaksi Berhasil Dibaca!")
display(df.head(10)) # Menampilkan 10 data awal

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Log Transaksi Berhasil Dibaca!


,ID_Transaksi,Waktu_Transaksi,ID_Mesin,Nama_Produk,Kategori,Jumlah_Beli,Metode_Bayar,Total_Harga,Uang_Masuk,Status_Transaksi
0,TRX-VM-001,2026-09-01 08:10:12,VM-GEDUNG_A-L1,Coca Cola 330ml,Minuman,1,QRIS,8000,8000,SUCCESS
1,TRX-VM-002,02/09/2026 08:15,vm_gedung_a_l1,good day mocacinno,minuman,1,qris_gopay,7000,7000,Berhasil
2,TRX-VM-003,2026-09-02 08:20:00,VM-GEDUNG_B-L2,Pokka Green Tea 450ml,Minuman,2,Cash_Koin,16000,20000,success
3,TRX-VM-004,2026-09-03T08:25:30,VM-GEDUNG_A-L1,Chitato Sapi Panggang,Snack,-1,Tunai,-10000,10000,FAILED
4,TRX-VM-005,2026-09-03 08:30:00,VM-GEDUNG-C-L3,Nescafe Latte,Kopi,1,ShopeePay,Rp 12.000,12000,Gagal
5,TRX-VM-006,04-09-2026 08:35,VM-GEDUNG_A-L1,Aqua_600ml,Minuman,dua,QRIS,4000,4000,1
6,TRX-VM-007,2026-09-04 08:40:10,VM-GEDUNG_B-L2,Silverqueen 62g,Cokelat,1,E-Money,15000,NaN,PENDING
7,TRX-VM-008,2026-09-05 08:45:00,VM-GEDUNG_A-L1,Ultra Milk Cokelat,Minuman,1,Flazz,6000,6000,SUCCESS
8,TRX-VM-008,2026-09-05 08:45:00,VM-GEDUNG_A-L1,Ultra Milk Cokelat,Minuman,1,Flazz,6000,6000,SUCCESS
9,TRX-VM-010,2026-09-05 08:50:00,VM-GEDUNG-C-L3,-,Snack,1,Cash,5000,5000,Berhasil


#**2. PEMERIKSAAN DATA**

##**2.1 Mengecek struktur dataset**

Setelah di cek, data kolom awal transaksi vending machine masih bertipe data object sehingga harus dilakukan penyesuaian lagi agar bisa sesuai dengan inputannya.

In [25]:
# Melihat jumlah baris, kolom, dan tipe data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   ID_Transaksi      50 non-null     object
 1   Waktu_Transaksi   50 non-null     object
 2   ID_Mesin          50 non-null     object
 3   Nama_Produk       48 non-null     object
 4   Kategori          49 non-null     object
 5   Jumlah_Beli       50 non-null     object
 6   Metode_Bayar      50 non-null     object
 7   Total_Harga       50 non-null     object
 8   Uang_Masuk        48 non-null     object
 9   Status_Transaksi  50 non-null     object
dtypes: object(10)
memory usage: 4.0+ KB


##**2.2 Mengecek missing value**

Selanjutnya, dari data yang sudah ada akan dilakukan pengecekan data kosong dari setiap kolomnya untuk dilakukan pembersihan maupun permbaikan.

In [26]:
# Mengecek jumlah data kosong pada setiap kolom
print("Missing value setiap kolom :")
print(df.isnull().sum())

Missing value setiap kolom :
ID_Transaksi        0
Waktu_Transaksi     0
ID_Mesin            0
Nama_Produk         2
Kategori            1
Jumlah_Beli         0
Metode_Bayar        0
Total_Harga         0
Uang_Masuk          2
Status_Transaksi    0
dtype: int64


#**3. CLEANSING DATA TRANSAKSI**

##**3.1 Standarisasi ID Transaksi**

Pada ID Transaksi diberlakukan juga standarisasi data untuk menyeragamkan format ID nya seperti strip untuk menghapus spasi berlebih di awal dan akhir ID dan upper untuk mengubah semua huruf menjadi kapital.

In [27]:
# Membersihkan dan menyeragamkan format ID transaksi
df['ID_Transaksi'] = (
    df['ID_Transaksi']
    .astype(str)
    .str.strip()
    .str.upper()
)

##**3.2 Standarisasi Waktu Transaksi**

Untuk waktu transaksi, dilakukan pengubahan format tanggal pada waktu transaksi menjadi sesuai ketentuan semisal data awal 02/09/2026 08:15 menjadi 2026-09-02 08:15:00

In [28]:
# Mengubah berbagai format tanggal menjadi format datetime
def perbaiki_tanggal(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}', value):
        return pd.to_datetime(value, errors='coerce')

    return pd.to_datetime(value, errors='coerce', dayfirst=True)

df['Waktu_Transaksi'] = df['Waktu_Transaksi'].apply(perbaiki_tanggal)

##**3.3 Standarisasi ID Mesin**

Kode dibawah berfungsi untuk membersihkan dan menyeragamkan format ID mesin agar penulisannya konsisten, sekaligus menampilkan perbandingan 5 data sebelum dan sesudah standarisasi sebagai contoh hasil cleansing.

In [29]:
# Menampilkan contoh 5 data sebelum standarisasi
idmesin_sebelum = df['ID_Mesin'].head(5).copy()

# Membersihkan format ID mesin
df['ID_Mesin'] = (
    df['ID_Mesin']
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace('_', '-', regex=False)
)

# Menyeragamkan format ID mesin
mapping_mesin = {
    'VM-GEDUNG-A-L1': 'VM-GEDUNG_A-L1',
    'VM-GEDUNG-B-L2': 'VM-GEDUNG_B-L2',
    'VM-GEDUNG-C-L3': 'VM-GEDUNG_C-L3'
}

df['ID_Mesin'] = df['ID_Mesin'].replace(mapping_mesin)

# Menampilkan perbandingan 5 data sebelum dan sesudah
perbandingan = pd.DataFrame({
    'Sebelum': idmesin_sebelum.values,
    'Sesudah': df['ID_Mesin'].head(5).values
})

display(perbandingan)

,Sebelum,Sesudah
0,VM-GEDUNG_A-L1,VM-GEDUNG_A-L1
1,vm_gedung_a_l1,VM-GEDUNG_A-L1
2,VM-GEDUNG_B-L2,VM-GEDUNG_B-L2
3,VM-GEDUNG_A-L1,VM-GEDUNG_A-L1
4,VM-GEDUNG-C-L3,VM-GEDUNG_C-L3


#**4. CLEANSING DATA PRODUK**

##**4.1 Standarisasi Nama Produk**

Lalu untuk produk yang memiliki ketidaksamaan nama, juga diberlakukan standarisasi melalui mapping produk yang berisi pasangan nama produk yang tidak seragam dengan nama produk yang telah ditentukan sebagai format standar.

In [30]:
# Menyeragamkan berbagai penulisan nama produk
mapping_produk = {
    'Coca Cola 330ml': 'Coca Cola 330 ml',
    'Coca-Cola': 'Coca Cola 330 ml',
    'COCA COLA 330ML': 'Coca Cola 330 ml',
    'Aqua_600ml': 'Aqua 600 ml',
    'Aqua 600ml': 'Aqua 600 ml',
    'good day mocacinno': 'Good Day Mocacinno',
    'good day cappuccino': 'Good Day Cappuccino',
    'kopi kapal api': 'Kopi Kapal Api',
    'Yakuult Pack (5pcs)': 'Yakult Pack (5pcs)',
    'Chitato_Sapi_Panggang': 'Chitato Sapi Panggang',
    'Lays_Rumput_Laut': 'Lays Rumput Laut 68 g',
    'Lays Rumput Laut 68g': 'Lays Rumput Laut 68 g',
    'Pop_Mie_Ayam': 'Pop Mie Ayam',
    'taro net 36g': 'Taro Net Seaweed 36 g',
    'Taro Net Seaweed': 'Taro Net Seaweed 36 g',
    'Oreo_Red_Velvet': 'Oreo Red Velvet',
    'Ichitan_Thai_Tea': 'Ichitan Thai Tea'
}

df['Nama_Produk'] = df['Nama_Produk'].replace(mapping_produk)

##**4.2 Menghapus Produk Tidak Valid**

Jika ada nama produk yang kosong dibeberapa data transaksi maka akan dilakukan penghapusan data agar nanti hasil akhirnya benar-benar merupakan data cleansing yang datanya valid.

In [31]:
# Mendeteksi nama produk yang kosong atau tidak diketahui
produk_tidak_valid = (
    df['Nama_Produk'].isna() |
    df['Nama_Produk'].astype(str).str.strip().isin([
        '', '-', 'Unknown Product'
    ])
)

print("Produk tidak valid :", produk_tidak_valid.sum())

# Menampilkan data yang akan dihapus
display(df[produk_tidak_valid])

# Menghapus produk tidak valid
df = df[~produk_tidak_valid].copy()

Produk tidak valid : 4


,ID_Transaksi,Waktu_Transaksi,ID_Mesin,Nama_Produk,Kategori,Jumlah_Beli,Metode_Bayar,Total_Harga,Uang_Masuk,Status_Transaksi
9,TRX-VM-010,2026-09-05 08:50:00,VM-GEDUNG_C-L3,-,Snack,1,Cash,5000,5000,Berhasil
14,TRX-VM-015,2026-09-08 09:15:00,VM-GEDUNG_A-L1,NaN,Minuman,1,E-Money,5000,5000,Sukses
21,TRX-VM-022,2026-09-11 09:50:00,VM-GEDUNG_A-L1,NaN,NaN,1,QRIS,7000,7000,1
37,TRX-VM-038,2026-09-19 12:45:00,VM-GEDUNG_B-L2,Unknown Product,uncatego,1,Cash,5000,5000,Ditolak


##**4.3 Standarisasi Kategori**

Selanjutnya untuk kolom kategori yang mempunyai ketidaksamaan nama juga dilakukan standarisasi dengan berpaku ke mapping kategori produk sama seperti mapping nama produk.

In [32]:
# Menyeragamkan kategori produk
mapping_kategori = {
    'Minuman': 'Minuman',
    'minuman': 'Minuman',
    'Snack': 'Makanan Ringan',
    'Makanan': 'Makanan Ringan',
    'makanan ringan': 'Makanan Ringan',
    'Cokelat': 'Cokelat',
    'Kopi': 'Kopi',
    'uncatego': 'UNKNOWN'
}

df['Kategori'] = df['Kategori'].replace(mapping_kategori)

#**5. CLEANSING JUMLAH PEMBELIAN**

Pada kolom jumlah pembelian juga dilakukan menyeragamkan jumlah pembelian ke bentuk angka, mendeteksi jumlah pembelian yang tidak valid (kosong, kurang dari atau sama dengan 0, dan lebih dari 10), kemudian menghapus data yang tidak valid dan mengubah tipe datanya menjadi integer.

In [33]:
# Mengubah angka yang ditulis dengan huruf menjadi angka
angka_huruf = {
    'satu': 1, 'dua': 2, 'tiga': 3, 'empat': 4, 'lima': 5,
    'enam': 6, 'tujuh': 7, 'delapan': 8,
    'sembilan': 9, 'sepuluh': 10
}

def ubah_jumlah_beli(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    return angka_huruf.get(
        value,
        pd.to_numeric(value, errors='coerce')
    )

# Menerapkan standarisasi jumlah pembelian
df['Jumlah_Beli'] = df['Jumlah_Beli'].apply(ubah_jumlah_beli)

# Mendeteksi jumlah pembelian yang tidak valid
jumlah_tidak_valid = (
    df['Jumlah_Beli'].isna() |
    (df['Jumlah_Beli'] <= 0) |
    (df['Jumlah_Beli'] > 10)
)

print("Jumlah pembelian tidak valid :", jumlah_tidak_valid.sum())

# Menampilkan data yang tidak valid
display(df[jumlah_tidak_valid])

# Menghapus data tidak valid dan mengubah tipe data
df = df[~jumlah_tidak_valid].copy()
df['Jumlah_Beli'] = df['Jumlah_Beli'].astype(int)

Jumlah pembelian tidak valid : 5


,ID_Transaksi,Waktu_Transaksi,ID_Mesin,Nama_Produk,Kategori,Jumlah_Beli,Metode_Bayar,Total_Harga,Uang_Masuk,Status_Transaksi
3,TRX-VM-004,2026-09-03 08:25:30,VM-GEDUNG_A-L1,Chitato Sapi Panggang,Makanan Ringan,-1,Tunai,-10000,10000,FAILED
11,TRX-VM-012,2026-09-06 09:00:00,VM-GEDUNG_A-L1,Good Day Cappuccino,Kopi,9999,QRIS,7000,7000,0
13,TRX-VM-014,2026-09-07 09:10:00,VM-GEDUNG_B-L2,Lays Rumput Laut 68 g,Makanan Ringan,0,Cash_Koin,0,0,CANCELLED
34,TRX-VM-035,2026-09-18 12:00:00,VM-GEDUNG_B-L2,Tango Wafer Cokelat,Makanan Ringan,-1,Cash_Koin,-8000,10000,FAILED
47,TRX-VM-048,2026-09-24 15:15:00,VM-GEDUNG_C-L3,Cheetos Jagung Bakar,Makanan Ringan,0,QRIS,0,0,CANCELLED


#**6. CLEANSING METODE PEMBAYARAN**

Menyeragamkan penulisan metode pembayaran, mendeteksi metode pembayaran yang tidak valid, lalu menghapus data yang tidak valid dari dataset contohnya seperti metode FREE dan unknown yang tidak sesuai ketentuan yg ada.

In [34]:
# Menyeragamkan metode pembayaran
mapping_pembayaran = {
    'QRIS': 'QRIS',
    'qris': 'QRIS',
    'qris_gopay': 'QRIS',
    'Tunai': 'CASH',
    'Cash': 'CASH',
    'Cash_Koin': 'CASH',
    'E-Money': 'E-MONEY',
    'Flazz': 'FLAZZ',
    'ShopeePay': 'SHOPEEPAY',
    'FREE': 'FREE',
    '1': 'UNKNOWN'
}

df['Metode_Bayar'] = df['Metode_Bayar'].replace(mapping_pembayaran)

# Mendeteksi metode pembayaran yang tidak valid
pembayaran_tidak_valid = df['Metode_Bayar'].isin([
    'FREE', 'UNKNOWN'
])
print("Pembayaran tidak valid:", pembayaran_tidak_valid.sum())

# Menampilkan dan menghapus data tidak valid
display(df[pembayaran_tidak_valid])
df = df[~pembayaran_tidak_valid].copy()

Pembayaran tidak valid: 3


,ID_Transaksi,Waktu_Transaksi,ID_Mesin,Nama_Produk,Kategori,Jumlah_Beli,Metode_Bayar,Total_Harga,Uang_Masuk,Status_Transaksi
16,TRX-VM-017,2026-09-09 09:25:00,VM-GEDUNG_C-L3,BENG BENG,Cokelat,3,UNKNOWN,9000,10000,success
25,TRX-VM-026,2026-09-13 10:10:00,VM-GEDUNG_A-L1,Cadbury Dairy Milk,Cokelat,1,FREE,0,0,Sukses
41,TRX-VM-042,2026-09-21 13:45:00,VM-GEDUNG_C-L3,Good Day Mocacinno,Kopi,1,UNKNOWN,7000,10000,0


#**7. CLEANSING DATA HARGA**

##**7.1 Standarisasi Total Harga**

Untuk total harga, akan dilakukan standarisasi dengan menyeragamkan format harga menjadi angka, termasuk mengubah penulisan seperti Rp 10.000 atau sepuluh ribu menjadi nilai numerik agar dapat digunakan dalam proses analisis dan perhitungan.

In [35]:
# Mengubah format harga seperti "Rp 10.000" menjadi angka
def ubah_harga(value):
    if pd.isna(value):
        return np.nan

    # Mengubah data menjadi teks dan membersihkan spasi/huruf besar
    value = str(value).strip().lower()
    # Mengubah harga menjadi angka
    angka_huruf = {
        'lima ribu': 5000,
        'tujuh ribu': 7000,
        'sepuluh ribu': 10000
    }
    # Mengambil nilai angka dari harga yang ditulis dengan huruf
    if value in angka_huruf:
        return angka_huruf[value]
    # Menghapus format mata uang dan tanda pemisah angka
    value = value.replace('rp', '').replace('.', '').replace(',', '')
    # Menghapus karakter selain angka
    value = re.sub(r'[^0-9\-]', '', value)
    # Mengubah hasil menjadi angka
    return pd.to_numeric(value, errors='coerce')

# Menerapkan standarisasi harga
df['Total_Harga'] = df['Total_Harga'].apply(ubah_harga)

##**7.2 Menghapus Harga Tidak Valid**

Setelah dilakukan standarisasi, harga yang ada dilakukan pemfilteran kembali untuk menghilangkan harga yang tidak valid.

In [36]:
# Mendeteksi harga kosong atau kurang dari/sama dengan nol
harga_tidak_valid = (
    df['Total_Harga'].isna() |
    (df['Total_Harga'] <= 0)
)

print("Harga tidak valid :", harga_tidak_valid.sum())

# Menampilkan dan menghapus harga tidak valid
display(df[harga_tidak_valid])

df = df[~harga_tidak_valid].copy()
df['Total_Harga'] = df['Total_Harga'].astype(int)

Harga tidak valid : 2


,ID_Transaksi,Waktu_Transaksi,ID_Mesin,Nama_Produk,Kategori,Jumlah_Beli,Metode_Bayar,Total_Harga,Uang_Masuk,Status_Transaksi
17,TRX-VM-018,2026-09-09 09:30:00,VM-GEDUNG_B-L2,milkuat botol,Minuman,1,CASH,-4000,5000,Gagal
28,TRX-VM-029,2026-09-15 10:25:00,VM-GEDUNG_A-L1,Ades Air Mineral 600ml,Minuman,1,CASH,-3000,3000,FAILED


#**8. PENYESUAIAN OUTLIER HARGA**

Menentukan batas bawah dan batas atas nilai Total_Harga menggunakan metode IQR, lalu dipakai untuk mendeteksi data harga yang termasuk outlier.

In [37]:
# Menghitung batas outlier menggunakan metode IQR
Q1 = df['Total_Harga'].quantile(0.25)
Q3 = df['Total_Harga'].quantile(0.75)
IQR = Q3 - Q1

batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

print("Batas bawah :", batas_bawah)
print("Batas atas  :", batas_atas)

Batas bawah : -812.5
Batas atas  : 19687.5


In [38]:
# Mendeteksi harga yang termasuk outlier
outlier = (
    (df['Total_Harga'] < batas_bawah) |
    (df['Total_Harga'] > batas_atas)
)

print("Jumlah outlier :", outlier.sum())

# Menampilkan data outlier
display(
    df.loc[outlier, [
        'ID_Transaksi',
        'Nama_Produk',
        'Total_Harga'
    ]]
)
# Menghapus outlier
df = df[~outlier].copy()

Jumlah outlier : 2


,ID_Transaksi,Nama_Produk,Total_Harga
19,TRX-VM-020,Kusuka Singkong Rumput Laut,999999
42,TRX-VM-043,Silverqueen Chunky Bar,22000


#**9. CLEANSING & ENCRICHMENT DATA UANG MASUK**

##**9.1 Cleansing Data Uang Masuk**

Kemudian akan diseragamkan format Uang_Masuk menjadi angka, mengisi nilai 0 pada transaksi PENDING yang belum memiliki pembayaran, lalu mengubah tipe datanya menjadi integer.

In [39]:
# Membersihkan format Rupiah pada Uang_Masuk
df['Uang_Masuk'] = df['Uang_Masuk'].apply(ubah_harga)

# Mengisi Uang_Masuk = 0 untuk transaksi PENDING yang belum membayar
mask_pending = (
    df['Status_Transaksi'].eq('PENDING') &
    df['Uang_Masuk'].isna()
)

df.loc[mask_pending, 'Uang_Masuk'] = 0

# Mengubah tipe data menjadi integer
df['Uang_Masuk'] = df['Uang_Masuk'].astype(int)

##**9.2 Data Enrichment**

In [40]:
# Menambahkan harga satuan produk
df['Harga_Satuan'] = (
    df['Total_Harga'] / df['Jumlah_Beli']
).round().astype(int)

# Menambahkan hari transaksi
df['Hari'] = df['Waktu_Transaksi'].dt.day_name()

# Mengubah nama hari menjadi bahasa Indonesia
hari_indonesia = {
    'Monday': 'Senin',
    'Tuesday': 'Selasa',
    'Wednesday': 'Rabu',
    'Thursday': 'Kamis',
    'Friday': 'Jumat',
    'Saturday': 'Sabtu',
    'Sunday': 'Minggu'
}

df['Hari'] = df['Hari'].replace(hari_indonesia)

# Menampilkan hasil enrichment
display(
    df[
        ['Waktu_Transaksi', 'Nama_Produk', 'Jumlah_Beli',
         'Total_Harga', 'Harga_Satuan', 'Hari']
    ].head(5)
)

,Waktu_Transaksi,Nama_Produk,Jumlah_Beli,Total_Harga,Harga_Satuan,Hari
0,2026-09-01 08:10:12,Coca Cola 330 ml,1,8000,8000,Selasa
1,2026-09-02 08:15:00,Good Day Mocacinno,1,7000,7000,Rabu
2,2026-09-02 08:20:00,Pokka Green Tea 450ml,2,16000,8000,Rabu
4,2026-09-03 08:30:00,Nescafe Latte,1,12000,12000,Kamis
5,2026-09-04 08:35:00,Aqua 600 ml,2,4000,2000,Jumat


#**10. STANDARISASI STATUS TRANSAKSI**

Status transaksi yang memiliki ketidaksamaan bentuk juga akan disesuaikan dengan hasil akhir status yaitu SUCCESS, FAILED, PENDING, CANCELLED, dan REFUND.

In [41]:
# Menyeragamkan status transaksi
mapping_status = {
    'SUCCESS': 'SUCCESS',
    'success': 'SUCCESS',
    'Berhasil': 'SUCCESS',
    'Sukses': 'SUCCESS',
    '1': 'SUCCESS',

    'FAILED': 'FAILED',
    'Gagal': 'FAILED',
    'Ditolak': 'FAILED',
    '0': 'FAILED',

    'PENDING': 'PENDING',
    'Pending': 'PENDING',

    'CANCELLED': 'CANCELLED',
    'REFUND': 'REFUND'
}

df['Status_Transaksi'] = df['Status_Transaksi'].replace(mapping_status)

#**11. PENGECEKAN DUPLIKASI DATA**

Setelah proses cleansing dan standarisasi selesai, data yang ada akan dilakukan pengecekan kembali untuk melihat apa ada data duplikat yang tersisa dan menghapusnya.

In [42]:
# Mendeteksi ID transaksi yang duplikat
duplikat = df.duplicated(
    subset='ID_Transaksi',
    keep=False
)

# Menampilkan ID transaksi yang duplikat
print("Jumlah ID transaksi duplikat :", df.loc[duplikat, 'ID_Transaksi'].nunique())
print("\nID transaksi yang duplikat :")
print(df.loc[duplikat, 'ID_Transaksi'].unique())

# Menampilkan data yang memiliki ID duplikat
display(
    df.loc[duplikat, ['ID_Transaksi', 'Waktu_Transaksi', 'ID_Mesin', 'Nama_Produk']]
    .sort_values('ID_Transaksi')
)

# Menghapus duplikasi dan mempertahankan data pertama
df = df.drop_duplicates(
    subset='ID_Transaksi',
    keep='first'
).copy()

Jumlah ID transaksi duplikat : 1

ID transaksi yang duplikat :
['TRX-VM-008']


,ID_Transaksi,Waktu_Transaksi,ID_Mesin,Nama_Produk
7,TRX-VM-008,2026-09-05 08:45:00,VM-GEDUNG_A-L1,Ultra Milk Cokelat
8,TRX-VM-008,2026-09-05 08:45:00,VM-GEDUNG_A-L1,Ultra Milk Cokelat


#**12. REKAP HASIL AKHIR DATA CLEANSING**

Di akhir akan ditampilkan rekap data setelah dilakukan cleansing, dengan menampilkan data jumlah baris, jumlah missing value yang ditemukan, tipe data tiap kolom, dan data duplikasi yang sudah terselesaikan.

In [43]:
# Menampilkan perbandingan kondisi dataset sebelum dan sesudah cleansing
print("HASIL AKHIR DATA CLEANSING")
print("=" * 30)

print("Jumlah baris awal    :", baris_awal)
print("Jumlah baris akhir   :", len(df))

print("Missing value awal   :", missing_awal)
print("Missing value akhir  :", df.isnull().sum().sum())

print("Duplikasi awal       :", duplikat_awal)
print("Duplikasi akhir      :", df.duplicated().sum())

print("Jumlah kolom         :", len(df.columns))

print("\nTipe data akhir :")
print(df.dtypes)

# Menampilkan data yang sudah bersih
display(df)

HASIL AKHIR DATA CLEANSING
Jumlah baris awal    : 50
Jumlah baris akhir   : 33
Missing value awal   : 5
Missing value akhir  : 0
Duplikasi awal       : 1
Duplikasi akhir      : 0
Jumlah kolom         : 12

Tipe data akhir :
ID_Transaksi                object
Waktu_Transaksi     datetime64[ns]
ID_Mesin                    object
Nama_Produk                 object
Kategori                    object
Jumlah_Beli                  int64
Metode_Bayar                object
Total_Harga                  int64
Uang_Masuk                   int64
Status_Transaksi            object
Harga_Satuan                 int64
Hari                        object
dtype: object


,ID_Transaksi,Waktu_Transaksi,ID_Mesin,Nama_Produk,Kategori,Jumlah_Beli,Metode_Bayar,Total_Harga,Uang_Masuk,Status_Transaksi,Harga_Satuan,Hari
0,TRX-VM-001,2026-09-01 08:10:12,VM-GEDUNG_A-L1,Coca Cola 330 ml,Minuman,1,QRIS,8000,8000,SUCCESS,8000,Selasa
1,TRX-VM-002,2026-09-02 08:15:00,VM-GEDUNG_A-L1,Good Day Mocacinno,Minuman,1,QRIS,7000,7000,SUCCESS,7000,Rabu
2,TRX-VM-003,2026-09-02 08:20:00,VM-GEDUNG_B-L2,Pokka Green Tea 450ml,Minuman,2,CASH,16000,20000,SUCCESS,8000,Rabu
4,TRX-VM-005,2026-09-03 08:30:00,VM-GEDUNG_C-L3,Nescafe Latte,Kopi,1,SHOPEEPAY,12000,12000,FAILED,12000,Kamis
5,TRX-VM-006,2026-09-04 08:35:00,VM-GEDUNG_A-L1,Aqua 600 ml,Minuman,2,QRIS,4000,4000,SUCCESS,2000,Jumat
6,TRX-VM-007,2026-09-04 08:40:10,VM-GEDUNG_B-L2,Silverqueen 62g,Cokelat,1,E-MONEY,15000,0,PENDING,15000,Jumat
7,TRX-VM-008,2026-09-05 08:45:00,VM-GEDUNG_A-L1,Ultra Milk Cokelat,Minuman,1,FLAZZ,6000,6000,SUCCESS,6000,Sabtu
10,TRX-VM-011,2026-09-06 08:55:00,VM-GEDUNG_B-L2,Oreo Original 133g,Makanan Ringan,1,CASH,8000,10000,FAILED,8000,Minggu
12,TRX-VM-013,2026-09-07 09:05:00,VM-GEDUNG_C-L3,Pocari Sweat 500ml,Minuman,1,QRIS,9000,9000,SUCCESS,9000,Senin
15,TRX-VM-016,2026-09-08 09:20:00,VM-GEDUNG_A-L1,Teh Pucuk Harum,Minuman,2,FLAZZ,8000,10000,REFUND,4000,Selasa


#**13. MENYIMPAN DATA BERSIH**

Akhirnya setelah pembersihan semua data selesai, data log transaksi vending machine otomatis disimpan dengan menggunakan output_path untuk mengunduh data kembali.

In [44]:
# Menyimpan dataset bersih ke Google Drive
output_path = '/content/drive/My Drive/2418014_DATASET DATMIN/dataset_vending_machine_aftercleansing.xlsx'

df.to_excel(output_path, index=False)

print("Data Log Transaksi Vending Machine (Clean) Telah Disimpan di : ")
print(output_path)

Data Log Transaksi Vending Machine (Clean) Telah Disimpan di : 
/content/drive/My Drive/2418014_DATASET DATMIN/dataset_vending_machine_aftercleansing.xlsx
